# 📘 Notebook 18: Semantic Search and Retrieval-Augmented Generation

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module F: Language Models in Practice · Notebook 4 of 4 in this module · (18 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Show, with a measurement, where **keyword search** fails
- Explain what a **sentence embedding** is and use one for **semantic search**
- Evaluate a retriever with **recall at k**
- Explain why a language model on its own cannot answer questions about your documents
- Build a complete **retrieval-augmented generation (RAG)** pipeline: chunk, retrieve, prompt, generate
- Name the ways a RAG system fails and the safeguards a responsible one needs

> **Prerequisites:** Notebooks 08, 10, 15 and 17 of this course. Pretrained Transformer models and the Hugging Face library are introduced in *From Python to Deep Learning & Fine-Tuning*, Notebook 17.
>
> 🔭 **Why this notebook matters:** this is where the course comes together. The search engine of Notebook 08 could not see that *car* and *automobile* mean the same. The embeddings of Notebook 10 could, but only for single words. The language models of this module write fluently, and know nothing about your own documents. Combine them and you have the design behind most practical AI assistants today: find the relevant passages by **meaning**, then let a language model **answer from them**. You will build one from end to end.

> ℹ️ **Setup note.** The retrieval machinery in this notebook is built and tested with plain NumPy and scikit-learn. The cells that load pretrained models (a sentence encoder of about 90 MB and a language model of about 1 GB) are meant for Colab, where the needed libraries are already installed.

In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

## 1. A small knowledge base

To see what retrieval adds, we need documents that no language model has ever read. So we return to Marlow Bay, the invented town of Notebook 17, and write its residents' handbook: sixteen short passages of practical information. Since the town does not exist, the only way to answer a question about it is to **find the right passage**.

In [ ]:
handbook = [
    "The town library on Harbour Street is open every day. On weekdays it opens at nine in the morning and closes at ten in the evening. At weekends it closes at six.",
    "Library members may borrow up to twelve books for three weeks. Tools, musical instruments and telescopes can be borrowed for one week. Membership is free for residents.",
    "The ferry to Eastfield leaves the harbour at 7:30, 12:00 and 17:45 on weekdays. The crossing takes forty minutes. There is no ferry service on Sundays.",
    "The weekly market is held in Mill Square every Saturday from eight until two. Farmers from the surrounding villages sell vegetables, cheese, honey and bread.",
    "The recycling centre on Quarry Road accepts glass, paper, metal, batteries and old electrical appliances. It is open from Tuesday to Saturday. Garden waste is collected separately on Mondays.",
    "The public swimming pool is heated and has six lanes. Adult tickets cost four euros and children under twelve swim for half price. The pool is closed for maintenance during the first week of September.",
    "Bus line 4 connects the railway station with the harbour every twenty minutes. A single ticket costs 1.80 euros and can be bought from the driver. Night buses run on Fridays and Saturdays only.",
    "Parking in the town centre is limited to two hours between eight in the morning and six in the evening. Residents can apply for an annual permit at the town hall. Parking is free on Sundays.",
    "The Maritime Museum tells the story of fishing and shipbuilding in the bay. It is housed in the old customs building and is closed on Mondays. Admission is free on the first Sunday of each month.",
    "The health clinic on Garden Lane has a doctor on duty from eight until eight. Outside those hours, urgent cases are seen at the hospital in Eastfield. Appointments can be made by telephone or online.",
    "The Lantern Festival takes place on the last weekend of August. Hundreds of paper lanterns are released over the water at dusk. Hotels are usually full, so visitors are advised to book early.",
    "Repairs to the sea wall begin next spring and are expected to last eight months. During the work the coastal path between the lighthouse and the old pier will be closed to walkers.",
    "Free wireless internet is available in the library, the town hall and Mill Square. No password is needed, but each session is limited to two hours.",
    "Items found on buses, on the ferry or in public buildings are taken to the lost property office at the town hall. They are kept for three months before being given to charity.",
    "The sports centre offers courts for tennis, basketball and badminton. Courts must be reserved at least one day in advance. Equipment can be hired at the reception desk.",
    "Dogs are welcome on the north beach all year round. On the south beach they are not allowed between May and September. Owners must keep dogs on a lead in the town centre.",
]

print(len(handbook), "passages,", sum(len(passage.split()) for passage in handbook), "words")

And a set of questions that residents might ask, each with the number of the passage that answers it. The questions are written the way people speak, which is often **not** in the words the handbook uses.

In [ ]:
questions = [
    ("How late can I stay in the library on a Wednesday?", 0),
    ("Can I take a guitar home from the library?", 1),
    ("When is the last boat to Eastfield?", 2),
    ("Where can I buy fresh food from local producers?", 3),
    ("How do I get rid of a broken toaster?", 4),
    ("How much does it cost for a child to go swimming?", 5),
    ("What is the fare from the train station to the port?", 6),
    ("How long may I leave my car in the middle of town?", 7),
    ("Is there somewhere to learn about the history of the fishing boats?", 8),
    ("I feel ill at midnight, where should I go?", 9),
    ("When is the best time to see the lights on the sea?", 10),
    ("Can I still walk along the coast next summer?", 11),
    ("Where can I get online without paying?", 12),
    ("I left my umbrella on the bus, who has it?", 13),
    ("Do I need to book to play tennis?", 14),
    ("May I bring my puppy to the seaside in July?", 15),
]
print(len(questions), "questions")

## 2. Where keyword search stops

Our first retriever is the search engine of Notebook 08: TF-IDF vectors and cosine similarity. We write the ranking function so that it works with **any** kind of vectors, since we will change the vectors later and keep everything else.

In [ ]:
def normalise(vectors):
    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    lengths[lengths == 0] = 1
    return vectors / lengths

def rank(query_vector, passage_vectors):
    """Return the passage numbers from most to least similar, with their similarities."""
    similarities = normalise(passage_vectors) @ normalise(query_vector.reshape(1, -1))[0]
    order = np.argsort(-similarities)
    return order, similarities[order]

vectoriser = TfidfVectorizer(stop_words="english")
tfidf_passages = vectoriser.fit_transform(handbook).toarray()

def tfidf_search(question, how_many=3):
    order, similarities = rank(vectoriser.transform([question]).toarray()[0], tfidf_passages)
    return order[:how_many].tolist(), similarities[:how_many]

for question in ["When does the ferry leave?", "When is the last boat to Eastfield?", "How do I get rid of a broken toaster?"]:
    found, similarities = tfidf_search(question, 1)
    print(f"Q: {question}\n   similarity {similarities[0]:.2f}: {handbook[found[0]][:80]}...\n")

The first question shares the word *ferry* with the handbook, and the right passage comes back. The second says *boat*, and goes wrong: the only word it shares with the ferry passage is the name *Eastfield*, which also occurs in the passage about the health clinic, and that one wins. The third fails completely. A *broken toaster* is an *old electrical appliance*, and the two phrases have no word in common. To a bag of words they are unrelated. Look at the similarity: it is zero, so the "best" passage is simply the first one in the list.

### Measuring it
One example is an anecdote. The standard measure for a retriever is **recall at k**: for what share of the questions is the correct passage among the top $k$ results?

In [ ]:
def recall_at(search_function, k):
    hits = 0
    for question, answer in questions:
        found, similarities = search_function(question, k)
        if answer in found:
            hits += 1
    return hits / len(questions)

for k in [1, 3, 5]:
    print(f"TF-IDF recall at {k}: {recall_at(tfidf_search, k):.0%}")

In [ ]:
print("Questions for which the right passage is not ranked first:\n")
for question, answer in questions:
    found, similarities = tfidf_search(question, 1)
    if found[0] != answer:
        print(f" - {question}")

Read the failures. In each one the question and the passage express the same thing in different words: *boat* and *ferry*, *guitar* and *musical instruments*, *puppy* and *dogs*, *seaside* and *beach*, *get online* and *wireless internet*, *ill at midnight* and *urgent cases outside those hours*. This is the limitation we recorded at the end of Notebook 08, and it has been waiting for us ever since. People do not ask questions in the vocabulary of the documents that answer them.

## 3. Searching by meaning

### Intuition first
In Notebook 10, word embeddings placed *car* next to *automobile*. If we had one vector for a whole **sentence**, built so that sentences with the same meaning lie close together, we could search by meaning. A question and its answer would be neighbours even with no word in common.

### Formal definition
A **sentence embedding** model maps a piece of text of any length to a single vector of fixed size. Modern ones are Transformer encoders (the BERT family) that have been trained further on hundreds of millions of pairs of texts known to belong together, such as a question and its answer, or a sentence and its paraphrase. The training pulls the vectors of matching pairs together and pushes the vectors of unrelated texts apart.

Unlike the word vectors of Notebook 10, these embeddings are **contextual**: the model reads the whole sentence, so *bank* in *river bank* and *bank* in *bank account* contribute differently. The limitation of "one vector per word, whatever its sense" has finally gone.

**Semantic search** is then the search of Notebook 08 with different vectors: embed every passage once, embed the query, rank by cosine similarity. Our `rank` function does not need to change at all.

The model below, `all-MiniLM-L6-v2`, is small (about 90 MB), fast, and produces vectors of 384 numbers.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")
embedded_passages = encoder.encode(handbook)
print("Passage embeddings:", embedded_passages.shape)

def semantic_search(question, how_many=3):
    order, similarities = rank(encoder.encode([question])[0], embedded_passages)
    return order[:how_many].tolist(), similarities[:how_many]

for question in ["When is the last boat to Eastfield?", "How do I get rid of a broken toaster?", "May I bring my puppy to the seaside in July?"]:
    found, similarities = semantic_search(question, 1)
    print(f"Q: {question}\n   similarity {similarities[0]:.2f}: {handbook[found[0]][:80]}...\n")

In [ ]:
print(f"{'':<10} {'TF-IDF':>8} {'semantic':>10}")
for k in [1, 3, 5]:
    print(f"recall@{k:<3} {recall_at(tfidf_search, k):>8.0%} {recall_at(semantic_search, k):>10.0%}")

Compare the two columns. The questions that defeated keyword search were all cases of the same meaning in different words, which is exactly what sentence embeddings are trained to handle.

### Neither method makes the other obsolete
Semantic search has weaknesses of its own. It can be vague about **exact** strings: a product code, a surname, an error number, a rare technical term the encoder barely knows. Keyword search is precise about exactly those. For this reason, production systems usually run **both** and merge the two rankings, which is known as **hybrid search**. One of the exercises builds a simple version.

> ⚠️ **Common pitfalls**
>
> - Embedding the passages with one model and the queries with another. Vectors from different models live in unrelated spaces and cannot be compared.
> - Embedding very long documents whole. One vector cannot represent ten pages faithfully. Split long documents into passages first (Section 5).
> - Judging a retriever by a handful of queries you thought of yourself. Build a test set of questions with known answers, as we did, and measure recall.

## 4. What a language model does not know

### Intuition first
A language model knows what was in its training text, in a compressed and imperfect form. It does not know anything written after its training ended. It does not know your company's documents, your course notes or your e-mails. And it knows nothing about Marlow Bay, which does not exist.

What does it do when asked anyway? You saw in Notebooks 13 and 17 that a decoder always produces something. Let us ask a real one. **FLAN-T5** is an openly available model of moderate size that has been trained to follow instructions.

In [ ]:
from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-base")

def ask_model(prompt):
    return generator(prompt, max_new_tokens=60, do_sample=False)[0]["generated_text"]

for question in ["When does the library in Marlow Bay close on weekdays?",
                 "How much is a single bus ticket in Marlow Bay?",
                 "What is the capital of France?"]:
    print(f"Q: {question}\nA: {ask_model(question)}\n")

The general-knowledge question is answered correctly, from what the model absorbed in training. For the two questions about Marlow Bay it has nothing to go on, and it answers anyway: either with a confident guess or with something beside the point. This is **hallucination** again, and now you can see where it comes from. The model is not lying. It is doing the only thing it does, producing a likely continuation.

There are two ways to give a model knowledge it lacks. One is to continue training it on your documents, which is the **fine-tuning** covered in Notebook 18 of *From Python to Deep Learning & Fine-Tuning*. It is costly, has to be repeated whenever the documents change, and still gives no guarantee against invention. The other is much simpler: **put the relevant text in front of the model at the moment it is asked.**

## 5. Retrieval-augmented generation

### Intuition first
Think of the difference between a closed-book and an open-book examination. In the first, a student must rely on memory, and may misremember. In the second, the student looks up the relevant page and answers from it. **Retrieval-augmented generation (RAG)** turns the language model's exam into an open-book one.

### Formal definition
A RAG system answers a question in three steps:

1. **Retrieve.** Use a search method to find the few passages most relevant to the question.
2. **Augment.** Build a prompt that contains those passages together with the question, and an instruction to answer from them.
3. **Generate.** Let the language model write the answer.

```
 question ──► retriever ──► top passages ──┐
     │                                     ▼
     └───────────────────────────►  prompt = instruction + passages + question
                                           │
                                           ▼
                                    language model ──► answer (with its sources)
```

Before any of that, long documents must be cut into passages. This is called **chunking**.

### Step 0: chunking
Our handbook is already in short passages. A real document, such as a manual or a report, is not, so here is the standard recipe: cut the text into pieces of a fixed number of words, with some **overlap** between neighbours, so that a sentence falling on a boundary is whole in at least one piece.

In [ ]:
def chunk(text, size=40, overlap=10):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
        start += size - overlap
    return chunks

long_text = " ".join(handbook[:4])
pieces = chunk(long_text, size=40, overlap=10)
print(len(long_text.split()), "words ->", len(pieces), "chunks\n")
for piece in pieces[:3]:
    print("-", piece, "\n")

Each chunk begins with the last ten words of the one before. The size is a real trade-off. Small chunks are precise and may lack the context needed to understand them. Large chunks carry context and mix several subjects in one vector, which blurs the search.

### Steps 1 and 2: retrieve and build the prompt
The prompt is ordinary text. Its wording matters: we tell the model to use **only** the information supplied, and to say so when the answer is not there.

In [ ]:
def build_prompt(question, passage_numbers):
    information = "\n".join(f"[{number}] {handbook[number]}" for number in passage_numbers)
    return (
        "Answer the question using only the information below. "
        "If the answer is not in the information, say \"I do not know\".\n\n"
        f"Information:\n{information}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

question = "When does the ferry leave?"
found, similarities = tfidf_search(question, 2)
print(build_prompt(question, found))

That is all the "augmentation" there is: text placed in front of the question. Nothing about the model has changed.

### Step 3: generate
Now the complete pipeline, using the semantic retriever of Section 3 and the language model of Section 4. It returns the answer **and the passages it was given**, so that a reader can check.

In [ ]:
def answer(question, how_many=2):
    found, similarities = semantic_search(question, how_many)
    reply = ask_model(build_prompt(question, found))
    return reply, found

for question in ["When does the library in Marlow Bay close on weekdays?",
                 "How much is a single bus ticket in Marlow Bay?",
                 "I left my umbrella on the bus, who has it?",
                 "May I bring my puppy to the seaside in July?"]:
    reply, sources = answer(question)
    print(f"Q: {question}\nA: {reply}\n   sources: {sources}\n")

Compare these answers with the closed-book answers of Section 4. The model is the same, down to the last weight. It has not learned anything about Marlow Bay. It has been **shown** the relevant passage and asked to read.

Three properties follow from this design, and together they explain why it is so widely used.

- **The knowledge is in the documents, not in the model.** Change a passage of the handbook and the next answer changes. No retraining.
- **Answers can be traced.** The system can show which passages it used, and a person can verify the answer against them.
- **The model can decline.** When retrieval finds nothing relevant, a well-instructed model can say that it does not know.

## 6. Does it work? Measuring and stress-testing

A RAG system has two components, and each can fail independently. They must be evaluated separately.

**The retriever.** We already measure it, with recall at $k$. If the right passage is not retrieved, the generator has no chance.

**The generator.** Given the right passage, does it produce the right answer? A simple check is to list, for a few questions, a string that a correct answer must contain.

In [ ]:
checks = [
    ("When does the library close on weekdays?", "ten"),
    ("How much is a single bus ticket?", "1.80"),
    ("How long does the ferry crossing take?", "forty"),
    ("On which day is the weekly market held?", "saturday"),
    ("How long are found items kept at the lost property office?", "three months"),
    ("How many books may a library member borrow?", "twelve"),
]

correct = 0
for question, expected in checks:
    reply, sources = answer(question)
    ok = expected in reply.lower()
    correct += ok
    print(f"{'OK  ' if ok else 'MISS'} {question}  ->  {reply}")
print(f"\n{correct} of {len(checks)} answers contain the expected fact.")

### Questions with no answer
The harder test is a question the handbook **cannot** answer. A good system should say so.

In [ ]:
for question in ["Who is the mayor of Marlow Bay?",
                 "How many people live in Marlow Bay?",
                 "What is the telephone number of the health clinic?"]:
    reply, sources = answer(question)
    print(f"Q: {question}\nA: {reply}\n   passages supplied: {sources}\n")

Study the results. The retriever **always** returns its top passages, however irrelevant, because ranking does not include an option for "nothing fits". Whether the model then declines, as instructed, or builds an answer out of whatever it was handed, depends on the model and on the prompt. Small models follow such instructions less reliably than large ones. This is the most important failure of RAG systems in practice, and one of the exercises explores a defence: refuse to answer when the best similarity is too low.

### How RAG systems fail
| Failure | What happens | Safeguard |
|---|---|---|
| Retrieval miss | the right passage is not found, and the model answers from the wrong one | measure recall, use hybrid search, retrieve more passages |
| Unanswerable question | an answer is invented from irrelevant passages | a similarity threshold, a firm instruction to decline |
| Ignored context | the model answers from its own memory and not from the passages | clearer prompt, a stronger model, checking the answer against the sources |
| Outdated or wrong documents | the answer faithfully repeats a mistake | maintain the documents, show dates and sources |
| Instructions hidden in documents | a passage contains text such as *"ignore your instructions and..."*, and the model obeys it | treat retrieved text as data, never as commands, and limit what the system is able to do |
| Private information | a passage that the person asking should not see is retrieved and quoted | apply access rights **before** retrieval |

> 🧠 **A rule for building these systems responsibly.** Always show the sources. A system that answers *"The library closes at ten in the evening on weekdays [passage 0]"* can be checked in seconds. One that answers without sources asks to be trusted, and nothing in this course suggests that a language model has earned that.

## 7. The whole course in one picture

The first notebook asked a question: **how do we turn text into numbers without losing what it means?** Every module gave a better answer, and each kept what worked in the one before.

| Module | Its answer | What it added | What it still lacked |
|---|---|---|---|
| A: Text as Data | clean tokens, counted | a way in: text becomes units we can count | everything about meaning |
| B: The Structure of Language | automata, grammars, tags | structure, and the first models learned from data | hand-written rules do not scale |
| C: Counting Words | n-grams, TF-IDF, Naive Bayes | prediction, search and classification from counts | every word an isolated symbol |
| D: Meaning as Vectors | embeddings, topics | similar words become neighbours | word order, and words with several senses |
| E: Neural Networks for Text | LSTMs, seq2seq, CNNs | order, memory, and sequences in and out | long-range memory, hunger for labelled data |
| F: Language Models in Practice | pretrained models, retrieval | knowledge from unlabelled text, meaning in context, answers grounded in sources | reliability, which remains our responsibility |

Look at what the system of this notebook is made of. Its retriever is the cosine similarity of Notebook 08, applied to vectors that descend from Notebook 10. Its generator is a language model in the sense of Notebook 07, predicting one token at a time, in the pieces of Notebook 15, with the decoding of Notebook 15. Its architecture is the encoder and decoder of Notebook 13. We judged it with recall, as in Notebook 08, and we checked its output for unsupported claims, as in Notebook 17.

Nothing in it is magic. It is a small number of ideas (count, compare, predict, represent as vectors, learn from data) combined and scaled. That was the claim of this course, and you have now built each piece yourself.

---
## ✏️ Exercises

### Exercise 1 (Ranking by hand)
Three passages have the vectors `[1, 0, 1]`, `[0, 2, 0]` and `[2, 1, 2]`, and a query has the vector `[1, 0, 2]`. Compute the cosine similarity of the query with each passage on paper, and give the ranking. Check with the `rank` function.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
passages = np.array([[1.0, 0, 1], [0, 2, 0], [2, 1, 2]])
query = np.array([1.0, 0, 2])

# passage 0: (1 + 0 + 2) / (sqrt(2) * sqrt(5)) = 3 / 3.162 = 0.949
# passage 1: 0 / ...                              = 0.000
# passage 2: (2 + 0 + 4) / (3 * sqrt(5))          = 6 / 6.708 = 0.894
order, similarities = rank(query, passages)
print(order, np.round(similarities, 3))
```

*Passage 2 has the larger dot product with the query, and passage 0 still ranks first, because cosine similarity compares directions and ignores length. This small function is the heart of every vector search system, however large.*
</details>

### Exercise 2 (Recall at every k)
Compute the recall of `tfidf_search` for every $k$ from 1 to 8 and print it. At what value of $k$ does it stop improving quickly? What does retrieving more passages cost in a RAG system?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for k in range(1, 9):
    recall = recall_at(tfidf_search, k)
    print(f"k = {k}: {recall:>4.0%} {'#' * int(recall * 40)}")
```

*Recall can only rise with k, and it rises more and more slowly. Each extra passage makes the prompt longer, which costs tokens (Notebook 15) and gives the model more irrelevant text to be distracted by. The usual practice is to retrieve a generous number, and then keep only the best few for the prompt.*
</details>

### Exercise 3 (Chunk sizes)
Join the whole handbook into one long text and split it with `chunk` using sizes 15, 40 and 100 words (overlap 5). For each, build a TF-IDF index over the chunks and find the best chunk for the question `"How much does it cost for a child to go swimming?"`. Print the chunk. Which size gives the most useful result?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
whole = " ".join(handbook)
question = "How much does it cost for a child to go swimming?"

for size in [15, 40, 100]:
    pieces = chunk(whole, size=size, overlap=5)
    indexer = TfidfVectorizer(stop_words="english")
    vectors = indexer.fit_transform(pieces).toarray()
    order, similarities = rank(indexer.transform([question]).toarray()[0], vectors)
    print(f"size {size:>3}: {len(pieces):>2} chunks, best similarity {similarities[0]:.2f}\n   {pieces[order[0]]}\n")
```

*Very small chunks may contain the matching words and cut off the fact that answers the question. Very large ones contain the answer buried among unrelated passages. Chunks close to the natural size of one topic work best, which is why splitting along paragraphs or headings, where a document has them, usually beats splitting by a fixed number of words.*
</details>

### Exercise 4 (Keep the handbook up to date)
The town council announces: *"From January, the library will also open a small branch at the railway station, where returned books can be left at any hour."* Add this as a new passage, rebuild the TF-IDF index, and check that the question `"Is there a library branch at the railway station?"` now finds it. What had to be retrained? Then try `"Where can I return a book at night?"` as well.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
updated = handbook + ["From January, the library will also open a small branch at the railway station, where returned books can be left at any hour."]

new_vectoriser = TfidfVectorizer(stop_words="english")
new_vectors = new_vectoriser.fit_transform(updated).toarray()

for question in ["Is there a library branch at the railway station?", "Where can I return a book at night?"]:
    order, similarities = rank(new_vectoriser.transform([question]).toarray()[0], new_vectors)
    print(f"Q: {question}\n   similarity {similarities[0]:.2f}: {updated[order[0]]}\n")
```

*Nothing was retrained. The new knowledge is available the moment the passage is indexed. With a semantic retriever you would embed just the one new passage and append its vector. This is the practical advantage of keeping knowledge in documents and not in the weights of a model. The second question shows the keyword retriever missing the new passage all the same: "return a book at night" shares no exact word with "returned books can be left at any hour", and the word "night" leads it to the night buses. Adding knowledge is easy. Finding it still depends on the retriever.*
</details>

### Exercise 5 (Hybrid search)
This exercise needs the `encoder` of Section 3, so run it in Colab. Combine the two retrievers: for each passage, add its **rank** in the TF-IDF results to its rank in the semantic results, and order the passages by that sum, smallest first. Write `hybrid_search(question, how_many)` and compare its recall at 1 and at 3 with the two separate methods.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def hybrid_search(question, how_many=3):
    by_keywords, similarities = rank(vectoriser.transform([question]).toarray()[0], tfidf_passages)
    by_meaning, similarities = rank(encoder.encode([question])[0], embedded_passages)

    total_rank = np.zeros(len(handbook))
    for position, passage in enumerate(by_keywords):
        total_rank[passage] += position
    for position, passage in enumerate(by_meaning):
        total_rank[passage] += position

    order = np.argsort(total_rank)
    return order[:how_many].tolist(), -total_rank[order[:how_many]]

print(f"{'':<10} {'TF-IDF':>8} {'semantic':>10} {'hybrid':>8}")
for k in [1, 3]:
    print(f"recall@{k:<3} {recall_at(tfidf_search, k):>8.0%} {recall_at(semantic_search, k):>10.0%} {recall_at(hybrid_search, k):>8.0%}")
```

*Adding ranks, and not similarity scores, avoids the problem that the two methods produce numbers on different scales. On a small set of paraphrased questions like ours, the semantic retriever is already strong and the combination may add little. Hybrid search earns its place on queries with exact names, codes and rare terms, which our test set does not contain. A method should be judged on the kind of queries it will really receive.*
</details>

### Exercise 6 (Learn to say "I do not know", a little harder)
This exercise needs the models of Sections 3 and 4, so run it in Colab. Write `careful_answer(question, threshold)` that retrieves the best passage and, if its similarity is below the threshold, returns `"I do not know."` **without calling the language model**. Otherwise it answers as before. Print the best similarity for the answerable questions in `questions` and for the three unanswerable ones of Section 6, choose a threshold, and test it.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
unanswerable = ["Who is the mayor of Marlow Bay?", "How many people live in Marlow Bay?", "What is the telephone number of the health clinic?"]

answerable_scores = [semantic_search(question, 1)[1][0] for question, passage in questions]
unanswerable_scores = [semantic_search(question, 1)[1][0] for question in unanswerable]
print("Answerable:   lowest", round(float(min(answerable_scores)), 2), " average", round(float(np.mean(answerable_scores)), 2))
print("Unanswerable: highest", round(float(max(unanswerable_scores)), 2), " average", round(float(np.mean(unanswerable_scores)), 2))

def careful_answer(question, threshold):
    found, similarities = semantic_search(question, 2)
    if similarities[0] < threshold:
        return "I do not know."
    return ask_model(build_prompt(question, found))

threshold = (min(answerable_scores) + max(unanswerable_scores)) / 2
print("Threshold:", round(float(threshold), 2), "\n")
for question in unanswerable + ["When does the library close on weekdays?"]:
    print(f"Q: {question}\nA: {careful_answer(question, threshold)}\n")
```

*Look at the two groups of scores before trusting the threshold. If the lowest answerable score is above the highest unanswerable one, a threshold between them separates the groups perfectly on this data. Often the groups overlap: the question about the clinic's telephone number retrieves the clinic passage with a high similarity, because it is on the right topic, and only lacks the fact. No threshold can catch that case. Similarity tells you that a passage is about the same subject, not that it contains the answer. Declining reliably remains one of the open problems of these systems.*
</details>

## 🔑 Key takeaways

- **Keyword search** fails when the question and the document use different words for the same thing, which is the normal case.
- A **sentence embedding** maps a whole text to one vector, with meaning-alike texts close together. **Semantic search** is cosine ranking over those vectors.
- Evaluate a retriever with **recall at k** on questions with known answers. Combine keyword and semantic search for the best of both.
- A language model knows only its training text, and when it does not know, it **invents**.
- **RAG** retrieves relevant passages, puts them in the prompt, and lets the model answer from them. The knowledge stays in the documents, can be updated instantly, and can be cited.
- RAG fails when retrieval misses, when the question has no answer, or when the documents are wrong. Evaluate the retriever and the generator **separately**.
- **Always show the sources.**

### 🎓 Congratulations: you have completed the entire course!

You began with a single word, eight characters in a row, and looked at the numbers behind its letters. You finish with a system that reads a collection of documents, finds the passage that answers a question asked in other words, and writes a reply that cites its source.

On the way you built, by hand, a tokeniser, a finite automaton, a parser, a part-of-speech tagger with the Viterbi algorithm, a language model, a search engine, a Naive Bayes classifier, word embeddings two ways, a topic model, an LSTM cell, a translator, a text CNN, a byte-pair encoder, a beam search and a summariser. You measured every one of them. Sometimes the measurement favoured the newer method and sometimes the simpler one, and each time you worked out why: what the task required, and how much data there was to learn it from. Keep that habit above all the others.

Language technology is moving quickly, and the systems of five years from now will differ from today's. The questions will be the same ones you have learned to ask. What are the units? How are they turned into numbers? What was this trained on, and to predict what? How was it evaluated, and against which baseline? What does it do when it does not know?

Thank you for working through these notebooks. The best next step is a project of your own: choose some text you care about, and find out what is in it.

---
*End of Module F: and the end of the course.*

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*